# 📊 Data Visualization dengan Matplotlib & Seaborn
**Fajri | Data Science Portfolio Project 02**

---

## 🎯 Tujuan Proyek
Menguasai teknik **Exploratory Data Analysis (EDA)** melalui 7 jenis visualisasi menggunakan Matplotlib dan Seaborn — dari yang paling dasar hingga yang lebih kompleks.

## 📋 Isi Notebook
1. 📦 Import Library & Konfigurasi
2. 🏗️ Generate Dataset
3. 📈 Chart 1: Line Chart + Moving Average
4. 📊 Chart 2: Bar Chart (Vertikal & Horizontal)
5. 🌡️ Chart 3: Heatmap Korelasi
6. 📉 Chart 4: Histogram + KDE
7. 📦 Chart 5: Box Plot (Deteksi Outlier)
8. 🔵 Chart 6: Scatter Plot
9. 🔗 Chart 7: Pair Plot
10. 🖼️ Dashboard Summary (semua chart dalam 1 gambar)

> ✅ **Cara pakai:** Jalankan setiap cell dari atas ke bawah dengan `Shift + Enter`.

## 1. 📦 Import Library & Konfigurasi

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Konfigurasi tampilan global ───────────────────────────────────
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams.update({
    'figure.dpi':       120,
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
    'axes.labelsize':   11,
    'xtick.labelsize':  9,
    'ytick.labelsize':  9,
})

print('✅ Library siap!')
print(f'   Matplotlib: {plt.matplotlib.__version__}')
print(f'   Seaborn   : {sns.__version__}')
print(f'   Pandas    : {pd.__version__}')

## 2. 🏗️ Generate Dataset

Kita akan membuat dataset penjualan toko yang lebih kaya fitur untuk keperluan visualisasi.

In [ ]:
np.random.seed(42)
N = 500

dates     = pd.date_range('2023-01-01', periods=N, freq='D')
kategori  = np.random.choice(['Elektronik', 'Fashion', 'Makanan', 'Olahraga'], N, p=[0.25, 0.35, 0.25, 0.15])
is_promo  = np.random.choice([0, 1], N, p=[0.70, 0.30])
is_wknd   = (dates.dayofweek >= 5).astype(int)

# Simulasi penjualan per kategori
base_map  = {'Elektronik': 80, 'Fashion': 60, 'Makanan': 45, 'Olahraga': 35}
base_vals = np.array([base_map[k] for k in kategori])
promo_eff = is_promo * np.random.uniform(10, 30, N)
seasonal  = 8 * np.sin(2 * np.pi * dates.dayofyear / 365)
noise     = np.random.normal(0, 8, N)

sales     = (base_vals + seasonal + promo_eff + is_wknd * 10 + noise).clip(5).round().astype(int)
revenue   = sales * np.random.uniform(50000, 200000, N).round(-3)
rating    = np.clip(np.random.normal(4.0, 0.5, N), 1.0, 5.0).round(1)
umur_pelanggan = np.random.randint(18, 65, N)

df = pd.DataFrame({
    'tanggal':   dates,
    'kategori':  kategori,
    'penjualan': sales,
    'revenue':   revenue,
    'is_promo':  is_promo,
    'is_weekend':is_wknd,
    'rating':    rating,
    'umur':      umur_pelanggan,
    'bulan':     dates.month,
    'hari':      dates.dayofweek,
})

print('✅ Dataset berhasil dibuat!')
print(f'   Shape: {df.shape}')
df.head()

## 3. 📈 Chart 1: Line Chart + Moving Average

**Kapan dipakai:** Melihat tren data dari waktu ke waktu.

**Teknik:** Overlay moving average untuk memperhalus noise dan melihat tren bersih.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

# Data harian (raw) — tampilan lebih transparent
ax.plot(df['tanggal'], df['penjualan'],
        color='steelblue', alpha=0.25, linewidth=0.7, label='Penjualan Harian')

# Moving average 7 hari
ma7 = df['penjualan'].rolling(7).mean()
ax.plot(df['tanggal'], ma7,
        color='steelblue', linewidth=2, label='7-Day Moving Average')

# Moving average 30 hari
ma30 = df['penjualan'].rolling(30).mean()
ax.plot(df['tanggal'], ma30,
        color='tomato', linewidth=2.5, linestyle='--', label='30-Day Moving Average')

# Anotasi hari promo
promo_days = df[df['is_promo'] == 1]
ax.scatter(promo_days['tanggal'], promo_days['penjualan'],
           color='gold', s=8, alpha=0.4, label='Hari Promo', zorder=3)

ax.set_title('Tren Penjualan Harian + Moving Average', fontsize=14)
ax.set_xlabel('Tanggal')
ax.set_ylabel('Unit Terjual')
ax.legend(loc='upper left')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig('chart1_line.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Insight: Titik emas menandai hari promo — terlihat jelas meningkatkan penjualan!')

## 4. 📊 Chart 2: Bar Chart

**Kapan dipakai:** Membandingkan nilai antar kategori.

**Teknik:** Grouped bar chart untuk membandingkan dua kondisi (promo vs non-promo) per kategori.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Bar Chart 1: Total penjualan per kategori ────────────────────
ax = axes[0]
cat_total = df.groupby('kategori')['penjualan'].sum().sort_values(ascending=False)
colors = sns.color_palette('husl', len(cat_total))
bars = ax.bar(cat_total.index, cat_total.values, color=colors, edgecolor='white', linewidth=1.2)
for bar, val in zip(bars, cat_total.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{val:,.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('Total Penjualan per Kategori')
ax.set_xlabel('Kategori')
ax.set_ylabel('Total Unit Terjual')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))

# ── Bar Chart 2: Grouped — Promo vs Non-Promo per Kategori ───────
ax = axes[1]
promo_cat = df.groupby(['kategori', 'is_promo'])['penjualan'].mean().unstack()
promo_cat.columns = ['Non-Promo', 'Promo']
x = np.arange(len(promo_cat))
w = 0.35
b1 = ax.bar(x - w/2, promo_cat['Non-Promo'], w, label='Non-Promo', color='#475569', edgecolor='white')
b2 = ax.bar(x + w/2, promo_cat['Promo'],     w, label='Promo',     color='#38bdf8', edgecolor='white')
ax.set_title('Rata-rata Penjualan: Promo vs Non-Promo')
ax.set_xticks(x)
ax.set_xticklabels(promo_cat.index)
ax.set_ylabel('Rata-rata Unit')
ax.legend()

plt.suptitle('Analisis Penjualan per Kategori', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('chart2_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Insight: Promo secara konsisten meningkatkan rata-rata penjualan di SEMUA kategori!')

## 5. 🌡️ Chart 3: Heatmap Korelasi

**Kapan dipakai:** Melihat hubungan linear antar banyak variabel sekaligus.

**Teknik:** Heatmap triangular (setengah saja) agar tidak redundan.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

num_cols = ['penjualan', 'revenue', 'is_promo', 'is_weekend', 'rating', 'umur', 'bulan', 'hari']
corr = df[num_cols].corr()

# Mask segitiga atas agar tidak duplikat
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', ax=ax,
    linewidths=0.5, linecolor='white',
    vmin=-1, vmax=1, square=True,
    annot_kws={'size': 10, 'weight': 'bold'},
    cbar_kws={'label': 'Koefisien Korelasi'}
)

ax.set_title('Heatmap Korelasi Antar Variabel', fontsize=14)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

plt.tight_layout()
plt.savefig('chart3_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Tampilkan korelasi tertinggi dengan target
print('🔍 Korelasi dengan variabel penjualan (diurutkan):')
corr_target = corr['penjualan'].drop('penjualan').abs().sort_values(ascending=False)
for col, val in corr_target.items():
    sign = '+' if corr['penjualan'][col] > 0 else '-'
    print(f'   {col:15s}: {sign}{val:.3f}')

## 6. 📉 Chart 4: Histogram + KDE

**Kapan dipakai:** Melihat distribusi satu variabel — apakah normal, skewed, atau bimodal.

**Teknik:** Overlay KDE (Kernel Density Estimation) untuk kurva distribusi yang halus.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Histogram penjualan keseluruhan ──────────────────────────────
ax = axes[0]
sns.histplot(df['penjualan'], bins=35, kde=True, ax=ax, color='steelblue',
             line_kws={'linewidth': 2.5})
ax.axvline(df['penjualan'].mean(),   color='red',    linestyle='--', lw=1.5,
           label=f'Mean: {df["penjualan"].mean():.1f}')
ax.axvline(df['penjualan'].median(), color='orange', linestyle='--', lw=1.5,
           label=f'Median: {df["penjualan"].median():.1f}')
ax.set_title('Distribusi Penjualan Harian')
ax.set_xlabel('Unit Terjual')
ax.set_ylabel('Frekuensi')
ax.legend()

# ── Histogram terpisah per kategori (KDE overlay) ────────────────
ax = axes[1]
for kat in df['kategori'].unique():
    subset = df[df['kategori'] == kat]['penjualan']
    sns.kdeplot(subset, ax=ax, label=kat, linewidth=2, fill=True, alpha=0.15)
ax.set_title('Distribusi Penjualan per Kategori (KDE)')
ax.set_xlabel('Unit Terjual')
ax.set_ylabel('Density')
ax.legend(title='Kategori')

plt.suptitle('Analisis Distribusi Penjualan', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('chart4_histogram.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'💡 Skewness penjualan: {df["penjualan"].skew():.3f}')
print('   (nilai > 0 = right-skewed, ada outlier di sisi kanan)')

## 7. 📦 Chart 5: Box Plot (Deteksi Outlier)

**Kapan dipakai:** Melihat distribusi, quartile, dan outlier antar kelompok.

**Teknik:** Violin plot + box plot digabung untuk informasi yang lebih lengkap.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── Box Plot per Kategori ─────────────────────────────────────────
ax = axes[0]
sns.boxplot(data=df, x='kategori', y='penjualan', ax=ax,
            palette='husl', width=0.5, linewidth=1.5,
            flierprops=dict(marker='o', markerfacecolor='red', markersize=4, alpha=0.5))
ax.set_title('Box Plot Penjualan per Kategori')
ax.set_xlabel('Kategori Produk')
ax.set_ylabel('Unit Terjual')

# Anotasi median
medians = df.groupby('kategori')['penjualan'].median()
for i, (kat, med) in enumerate(medians.items()):
    ax.text(i, med + 1, f'  {med:.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# ── Violin Plot: Promo vs Non-Promo ──────────────────────────────
ax = axes[1]
df['Kondisi'] = df['is_promo'].map({0: 'Non-Promo', 1: 'Promo'})
sns.violinplot(data=df, x='Kondisi', y='penjualan', ax=ax,
               palette=['#475569', '#38bdf8'], inner='quartile', linewidth=1.5)
ax.set_title('Violin Plot: Distribusi Promo vs Non-Promo')
ax.set_xlabel('')
ax.set_ylabel('Unit Terjual')

plt.suptitle('Analisis Distribusi & Outlier', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('chart5_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

# Hitung outlier dengan IQR
Q1, Q3 = df['penjualan'].quantile([0.25, 0.75])
IQR = Q3 - Q1
outliers = df[(df['penjualan'] < Q1 - 1.5*IQR) | (df['penjualan'] > Q3 + 1.5*IQR)]
print(f'🔍 Jumlah outlier terdeteksi: {len(outliers)} baris ({len(outliers)/len(df)*100:.1f}%)')
print(f'   Batas bawah: {Q1 - 1.5*IQR:.1f}  |  Batas atas: {Q3 + 1.5*IQR:.1f}')

## 8. 🔵 Chart 6: Scatter Plot

**Kapan dipakai:** Melihat hubungan dua variabel numerik + identifikasi cluster/pola.

**Teknik:** Color-coding kategori + regression line per kelompok.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── Scatter: Revenue vs Penjualan, warna per kategori ────────────
ax = axes[0]
palette = {'Elektronik': '#38bdf8', 'Fashion': '#f472b6',
           'Makanan': '#4ade80', 'Olahraga': '#fbbf24'}
for kat, color in palette.items():
    sub = df[df['kategori'] == kat]
    ax.scatter(sub['penjualan'], sub['revenue']/1e6, label=kat,
               color=color, alpha=0.5, s=30, edgecolors='white', linewidth=0.3)
ax.set_title('Hubungan Penjualan vs Revenue')
ax.set_xlabel('Unit Terjual')
ax.set_ylabel('Revenue (Juta Rp)')
ax.legend(title='Kategori', loc='upper left', fontsize=9)

# ── Scatter: Umur vs Rating, ukuran titik = penjualan ────────────
ax = axes[1]
sc = ax.scatter(df['umur'], df['rating'],
                c=df['penjualan'], cmap='YlOrRd',
                s=df['penjualan'] * 0.4, alpha=0.5, edgecolors='white', linewidth=0.3)
plt.colorbar(sc, ax=ax, label='Penjualan (unit)')
ax.set_title('Umur Pelanggan vs Rating (ukuran = penjualan)')
ax.set_xlabel('Umur Pelanggan')
ax.set_ylabel('Rating Produk')

plt.suptitle('Scatter Plot — Hubungan Antar Variabel', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('chart6_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Tip: Ukuran dan warna titik bisa digunakan untuk encode 2 variabel tambahan!')

## 9. 🔗 Chart 7: Pair Plot

**Kapan dipakai:** Melihat SEMUA kombinasi pasangan variabel sekaligus — sangat berguna di awal EDA.

**Teknik:** `sns.pairplot()` dengan hue untuk membedakan kategori.

In [ ]:
# Ambil subset kolom yang paling informatif
pair_cols = ['penjualan', 'revenue', 'rating', 'umur', 'is_promo', 'kategori']
df_pair = df[pair_cols].copy()
df_pair['revenue'] = df_pair['revenue'] / 1e6  # konversi ke Juta
df_pair['is_promo'] = df_pair['is_promo'].map({0: 'No', 1: 'Yes'})

g = sns.pairplot(
    df_pair,
    hue='kategori',
    vars=['penjualan', 'revenue', 'rating', 'umur'],
    diag_kind='kde',
    plot_kws={'alpha': 0.35, 's': 15},
    diag_kws={'linewidth': 2},
    palette='husl',
    height=2.5
)
g.fig.suptitle('Pair Plot — Semua Kombinasi Variabel', y=1.02, fontsize=14, fontweight='bold')
plt.savefig('chart7_pairplot.png', dpi=120, bbox_inches='tight')
plt.show()
print('💡 Diagonal: distribusi tiap variabel per kategori')
print('   Off-diagonal: scatter plot semua pasangan kombinasi')

## 10. 🖼️ Dashboard Summary

Menggabungkan 4 chart utama dalam satu gambar untuk keperluan presentasi atau laporan.

In [ ]:
fig = plt.figure(figsize=(18, 12))
fig.patch.set_facecolor('#0f172a')
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

title_kw = dict(fontsize=12, fontweight='bold', color='white', pad=10)
label_kw = dict(color='#94a3b8', fontsize=9)

# ── Panel 1: Line Chart (lebar 2 kolom) ──────────────────────────
ax1 = fig.add_subplot(gs[0, :2])
ax1.set_facecolor('#1e293b')
ax1.plot(df['tanggal'], df['penjualan'].rolling(7).mean(),
         color='#38bdf8', linewidth=2, label='7-Day MA')
ax1.plot(df['tanggal'], df['penjualan'].rolling(30).mean(),
         color='#f472b6', linewidth=2, linestyle='--', label='30-Day MA')
ax1.set_title('Tren Penjualan', **title_kw)
ax1.tick_params(colors='#94a3b8')
ax1.legend(fontsize=9, facecolor='#1e293b', labelcolor='white')
for spine in ax1.spines.values(): spine.set_color('#334155')

# ── Panel 2: Bar Chart ────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
ax2.set_facecolor('#1e293b')
cat_total = df.groupby('kategori')['penjualan'].sum().sort_values()
colors_bar = ['#38bdf8', '#818cf8', '#4ade80', '#fbbf24']
ax2.barh(cat_total.index, cat_total.values, color=colors_bar)
ax2.set_title('Total per Kategori', **title_kw)
ax2.tick_params(colors='#94a3b8')
for spine in ax2.spines.values(): spine.set_color('#334155')

# ── Panel 3: Heatmap Korelasi ─────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
ax3.set_facecolor('#1e293b')
corr_small = df[['penjualan','revenue','is_promo','rating','umur']].corr()
mask = np.triu(np.ones_like(corr_small, dtype=bool))
sns.heatmap(corr_small, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', ax=ax3, linewidths=0.5, square=True,
            annot_kws={'size': 8, 'color': 'white'},
            cbar=False, vmin=-1, vmax=1)
ax3.set_title('Korelasi', **title_kw)
ax3.tick_params(colors='#94a3b8', labelsize=8)

# ── Panel 4: Box Plot ─────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
ax4.set_facecolor('#1e293b')
sns.boxplot(data=df, x='kategori', y='penjualan', ax=ax4,
            palette='husl', width=0.5, linewidth=1)
ax4.set_title('Distribusi per Kategori', **title_kw)
ax4.tick_params(colors='#94a3b8', labelsize=8)
ax4.set_xlabel('')
ax4.set_ylabel('Unit', color='#94a3b8')
for spine in ax4.spines.values(): spine.set_color('#334155')

# ── Panel 5: Histogram ────────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
ax5.set_facecolor('#1e293b')
sns.histplot(df['penjualan'], bins=30, kde=True, ax=ax5,
             color='#38bdf8', line_kws={'linewidth': 2})
ax5.set_title('Distribusi Penjualan', **title_kw)
ax5.tick_params(colors='#94a3b8')
ax5.set_xlabel('Unit', color='#94a3b8')
ax5.set_ylabel('Frekuensi', color='#94a3b8')
for spine in ax5.spines.values(): spine.set_color('#334155')

# Title keseluruhan
fig.text(0.5, 0.98, '📊 Sales Analytics Dashboard',
         ha='center', va='top', fontsize=18, fontweight='bold', color='white')
fig.text(0.5, 0.95, 'Fajri | Data Science Portfolio',
         ha='center', va='top', fontsize=11, color='#94a3b8')

plt.savefig('dashboard_summary.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()
print('💾 Dashboard disimpan: dashboard_summary.png')

## 📌 Ringkasan Teknik Visualisasi

| Chart | Library | Kapan Dipakai |
|---|---|---|
| Line Chart | Matplotlib | Data time series, tren |
| Bar Chart | Matplotlib | Perbandingan antar kategori |
| Heatmap | Seaborn | Korelasi banyak variabel |
| Histogram + KDE | Seaborn | Distribusi satu variabel |
| Box Plot | Seaborn | Distribusi + deteksi outlier |
| Scatter Plot | Matplotlib | Hubungan 2 variabel numerik |
| Pair Plot | Seaborn | Eksplorasi awal, semua kombinasi |

### 💡 Tips Penting
- Selalu tambahkan **judul, label sumbu, dan legenda** yang jelas
- Gunakan `plt.savefig('nama.png', dpi=150, bbox_inches='tight')` untuk menyimpan kualitas tinggi
- Pilih jenis chart berdasarkan **tipe data** dan **pertanyaan** yang ingin dijawab

---
**© 2026 Fajri | Data Science Portfolio**